## Query Items using SQL in Azure CosmosDB with Python SDK

### Installing Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv

### Setting up the Environment

In [ ]:
import os 
from dotenv import load_dotenv
import json

load_dotenv()

endpoint = os.getenv("COSMOSDB_ENDPOINT")
key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient

client = CosmosClient(endpoint, key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Write Basic SELECT Queries

In [ ]:
query = """SELECT
    c.id,
    c.name,
    c.category,
    c.priceValue
FROM c"""

items = container.query_items(query=query, enable_cross_partition_query=True)

for item in items:
    print(f"Name: {item['name']}, Category: {item['category']}, Price: {item['priceValue']}")

### Use a WHERE Clause

In [ ]:
query = """SELECT *
FROM c
WHERE c.category = "Smoothies" """

items = container.query_items(query=query, enable_cross_partition_query=True)

for item in items:
    print(f"Name: {item['name']}, Category: {item['category']}, Price: {item['priceValue']}")

### Perform a Full-Text Search

In [ ]:
query = """SELECT *
FROM c
WHERE CONTAINS(c.description, "avocado") """

items = container.query_items(query=query, enable_cross_partition_query=True)

for item in items:
    print(f"Name: {item['name']}, Category: {item['category']}, Price: {item['priceValue']}")

### Calculate Aggregates

In [ ]:
query = """
SELECT VALUE AVG(c.priceValue)
FROM c
"""

average = container.query_items(query=query, enable_cross_partition_query=True)

print(f"Average Price: {list(average)[0]}")

In [ ]:
query = """
SELECT VALUE COUNT(1)
FROM c
WHERE c.category = "Smoothies"
"""

total_smoothies = container.query_items(
    query=query,
    partition_key="Smoothies"
)

print(f"Total Smoothies: {list(total_smoothies)[0]}")

### Query arrays within documents

In [ ]:
query = """ SELECT * FROM c
WHERE ARRAY_CONTAINS(c.dietaryTags, "High Protein") """

items = container.query_items(query=query, enable_cross_partition_query=True)
for item in items:
    print(f"Name: {item['name']}, Category: {item['category']}, Price: {item['priceValue']}, Dietary Tags: {item['dietaryTags']}")

### Use parameterized queries

In [ ]:
query = """ 
SELECT * FROM c
WHERE c.category = @category
AND c.priceValue < @price
ORDER BY c.priceValue DESC
"""

parameters = [
    {"name": "@category", "value": "Smoothies"},
    {"name": "@price", "value": 5.5}
]

items = container.query_items(
    query=query,
    parameters=parameters,
    partition_key="Smoothies"
)

for item in items:
    print(f"Name: {item['name']}, Category: {item['category']}, Price: {item['priceValue']}")


### Shape Results with Projections

In [ ]:
query = """ 
SELECT VALUE {
    "productName": p.name,
    "productCategory": p.category,
    "productPrice": p.priceValue
}
FROM products p
WHERE p.category = @category"""

parameters = [
    {"name": "@category", "value": "Smoothies"}
]

items = container.query_items(
    query=query,
    parameters=parameters,
    partition_key="Smoothies"
)

for item in items:
    print(f"Name: {item['productName']}, Category: {item['productCategory']}, Price: {item['productPrice']}")

CosmosHttpResponseError: (BadRequest) Message: {"errors":[{"severity":"Error","location":{"start":36,"end":37},"code":"SC2001","message":"Identifier 'c' could not be resolved."},{"severity":"Error","location":{"start":67,"end":68},"code":"SC2001","message":"Identifier 'c' could not be resolved."},{"severity":"Error","location":{"start":99,"end":100},"code":"SC2001","message":"Identifier 'c' could not be resolved."}]}
ActivityId: 384d27a2-ab1e-4f07-948f-1e17b5b7d03b, Request URI: /apps/497aa0ac-49c2-41d9-9ac9-f4cf41815b18/services/37c870f0-db02-49a3-8bb3-81e126056105/partitions/8c434836-a1ea-4b0e-9210-5025162dc917/replicas/134247872745273959s, RequestStats: , SDK: Microsoft.Azure.Documents.Common/2.14.0
Code: BadRequest
Message: Message: {"errors":[{"severity":"Error","location":{"start":36,"end":37},"code":"SC2001","message":"Identifier 'c' could not be resolved."},{"severity":"Error","location":{"start":67,"end":68},"code":"SC2001","message":"Identifier 'c' could not be resolved."},{